In [2]:
# prerequisites
import torch
import torch.nn as nn
import torch.nn.functional as F
import torch.optim as optim
from torchvision import datasets, transforms
from torch.autograd import Variable
from torchvision.utils import save_image

# Device configuration
device = torch.device('cuda' if torch.cuda.is_available() else 'cpu')
print(device)

cpu


In [ ]:
BATCH_SIZE = 100

# FASION MNIST Dataset
transform = transforms.Compose([
    transforms.ToTensor(),
    transforms.Normalize(0.13, 0.31)])

train_dataset = datasets.FashionMNIST(root='data', train=True, transform=transform, download=True)
test_dataset = datasets.FashionMNIST(root='data', train=False, transform=transform, download=True)

In [5]:
# Data Loader (Input Pipeline)
train_loader = torch.utils.data.DataLoader(dataset=train_dataset, batch_size=BATCH_SIZE, shuffle=True)
test_loader = torch.utils.data.DataLoader(dataset=test_dataset, batch_size=BATCH_SIZE, shuffle=False)

In [ ]:
hidden_dimension = 128

def init_weights(m):
    if type(m) == torch.nn.Linear:
        torch.nn.init.xavier_uniform_(m.weight)
        m.bias.data.fill_(0.01)

class Generator(nn.Module):
    def __init__(self, g_input_dim, g_output_dim):
        super(Generator, self).__init__()       
        self.fc1 = nn.Linear(g_input_dim, hidden_dimension)
        self.fc2 = nn.Linear(hidden_dimension, hidden_dimension)
        self.fc3 = nn.Linear(hidden_dimension, hidden_dimension)
        self.fc4 = nn.Linear(hidden_dimension, g_output_dim)

        
        self.apply(init_weights)
    
    # forward method
    def forward(self, x): 
        x = F.relu(self.fc1(x))
        x = F.relu(self.fc2(x))
        x = F.relu(self.fc3(x))
        return torch.tanh(self.fc4(x))
    
class Discriminator(nn.Module):
    def __init__(self, d_input_dim):
        super(Discriminator, self).__init__()
        self.fc1 = nn.Linear(d_input_dim, hidden_dimension)
        self.fc2 = nn.Linear(self.fc1.out_features, self.fc1.out_features)
        self.fc3 = nn.Linear(self.fc2.out_features, self.fc2.out_features)
        self.fc4 = nn.Linear(self.fc3.out_features, 1)
        
        self.apply(init_weights)
    
    # forward method
    def forward(self, x):
        x = F.leaky_relu(self.fc1(x), 0.2)
        x = F.dropout(x, 0.3)
        x = F.leaky_relu(self.fc2(x), 0.2)
        x = F.dropout(x, 0.3)
        x = F.leaky_relu(self.fc3(x), 0.2)
        x = F.dropout(x, 0.3)
        return torch.sigmoid(self.fc4(x))

In [7]:
# build network
Z_DIM = 50
mnist_dim = train_dataset.train_data.size(1) * train_dataset.train_data.size(2)

print (f'mnist_dim = {mnist_dim}')

G = Generator(g_input_dim = Z_DIM, g_output_dim = mnist_dim).to(device)
D = Discriminator(mnist_dim).to(device)

mnist_dim = 784


c:\Users\Manya\.virtualenvs\.venv\Lib\site-packages\torchvision\datasets\mnist.py:76: UserWarning: train_data has been renamed data
  warnings.warn("train_data has been renamed data")


In [8]:
G

Generator(
  (fc1): Linear(in_features=50, out_features=128, bias=True)
  (fc2): Linear(in_features=128, out_features=128, bias=True)
  (fc3): Linear(in_features=128, out_features=128, bias=True)
  (fc4): Linear(in_features=128, out_features=784, bias=True)
)

In [9]:
D

Discriminator(
  (fc1): Linear(in_features=784, out_features=128, bias=True)
  (fc2): Linear(in_features=128, out_features=128, bias=True)
  (fc3): Linear(in_features=128, out_features=128, bias=True)
  (fc4): Linear(in_features=128, out_features=1, bias=True)
)

In [10]:
# loss
criterion = nn.BCELoss() 

# optimizer
lr = 0.0002 
G_optimizer = optim.Adam(G.parameters(), lr = lr)
D_optimizer = optim.Adam(D.parameters(), lr = lr)

In [ ]:
def D_train(x):
    D.zero_grad()

    # train discriminator on real
    x_real, y_real = x.view(-1, mnist_dim), torch.ones(BATCH_SIZE, 1)
    x_real, y_real = Variable(x_real.to(device)), Variable(y_real.to(device))

    D_output = D(x_real)
    D_real_loss = criterion(D_output, y_real)
    D_real_score = D_output

    # train discriminator on fake
    z = Variable(torch.randn(BATCH_SIZE, Z_DIM).to(device))
    x_fake, y_fake = G(z), Variable(torch.zeros(BATCH_SIZE, 1).to(device))

    D_output = D(x_fake)
    D_fake_loss = criterion(D_output, y_fake)
    D_fake_score = D_output

    D_loss = D_real_loss + D_fake_loss
    D_loss.backward()
    D_optimizer.step()
        
    return  D_loss.data.item()

In [ ]:
def G_train(x):
    G.zero_grad()

    z = Variable(torch.randn(BATCH_SIZE, Z_DIM).to(device))
    y = Variable(torch.ones(BATCH_SIZE, 1).to(device))

    G_output = G(z)
    D_output = D(G_output)
    G_loss = criterion(D_output, y)

    G_loss.backward()
    G_optimizer.step()
        
    return G_loss.data.item()

In [15]:
test_z = Variable(torch.randn(BATCH_SIZE, Z_DIM).to(device))

def generate_test_image(epoch):
    with torch.no_grad():
        generated = G(test_z)
        save_image(generated.view(generated.size(0), 1, 28, 28)[0], 
                   f'output/sample_{epoch}.png')

In [16]:
n_epoch = 300
D_losses, G_losses = [], []

for epoch in range(1, n_epoch+1):
    for batch_idx, (x, _) in enumerate(train_loader):
        D_losses.append(D_train(x))
        G_losses.append(G_train(x))

    print('[%d/%d]: loss_d: %.3f, loss_g: %.3f' % (
            (epoch), n_epoch, torch.mean(torch.FloatTensor(D_losses)), torch.mean(torch.FloatTensor(G_losses))))

    generate_test_image(epoch)

[1/300]: loss_d: 0.225, loss_g: 4.270
[2/300]: loss_d: 0.228, loss_g: 4.415
[3/300]: loss_d: 0.206, loss_g: 4.446
[4/300]: loss_d: 0.202, loss_g: 4.394
[5/300]: loss_d: 0.191, loss_g: 4.506
[6/300]: loss_d: 0.191, loss_g: 4.517
[7/300]: loss_d: 0.183, loss_g: 4.565
[8/300]: loss_d: 0.179, loss_g: 4.607
[9/300]: loss_d: 0.175, loss_g: 4.679
[10/300]: loss_d: 0.174, loss_g: 4.732
[11/300]: loss_d: 0.172, loss_g: 4.714
[12/300]: loss_d: 0.175, loss_g: 4.670
[13/300]: loss_d: 0.176, loss_g: 4.608
[14/300]: loss_d: 0.177, loss_g: 4.566
[15/300]: loss_d: 0.177, loss_g: 4.523
[16/300]: loss_d: 0.176, loss_g: 4.487
[17/300]: loss_d: 0.176, loss_g: 4.459
[18/300]: loss_d: 0.174, loss_g: 4.448
[19/300]: loss_d: 0.173, loss_g: 4.434
[20/300]: loss_d: 0.171, loss_g: 4.425
[21/300]: loss_d: 0.169, loss_g: 4.413
[22/300]: loss_d: 0.168, loss_g: 4.408
[23/300]: loss_d: 0.166, loss_g: 4.399
[24/300]: loss_d: 0.165, loss_g: 4.403
[25/300]: loss_d: 0.163, loss_g: 4.404
[26/300]: loss_d: 0.161, loss_g: 4

In [17]:
generated = G(test_z)

In [18]:
generated.shape

torch.Size([100, 784])

In [19]:
save_image(generated.view(generated.size(0), 1, 28, 28), 
                   f'output/all.png')